# Training PPO tiny tackers
### Run on Google Colab for superior training efficiency and download model to models folder in Tiny-Tackers directory

- - - - - - - - - - - - - - - - - - - -
## Training a PPO Base Model on Gabo-Tor Environment
- Windward Buoy (only has to reach it)
- - - - - - - - - - - - - - - - - - - -

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Set directory to tiny_tackers
%cd "/content/drive/My Drive/tiny_tackers"

/content/drive/My Drive/tiny_tackers


In [ ]:
# Install and import dependencies
!pip install stable-baselines3 gymnasium pygame

import os
import sys
import time
import pandas as pd
import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy

from gymnasium.wrappers import RecordVideo

In [ ]:
# Set path to retrieving the base environment
BASE_ENV_PATH = "/content/drive/My Drive/tiny_tackers/gym_sailing_environments/gym_sailing_gabo-tor"
sys.path.append(BASE_ENV_PATH)

In [ ]:
# Import environment and ensure it's comming from the correct path. Ending in tiny_tackers/gym_sailing_environments/gym_sailing_gabo-tor/gym_sailing/__init__.py
import gym_sailing
print(gym_sailing.__file__)

/content/drive/My Drive/tiny_tackers/gym_sailing_environments/gym_sailing_gabo-tor/gym_sailing/__init__.py


/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:636: UserWarning: WARN: Overriding environment SailboatDiscrete-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:636: UserWarning: WARN: Overriding environment Motorboat-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


In [ ]:
# Specifify that the environment is going to be one where continuous actions are possible not discrete
ENV_ID = "Sailboat-v0"

In [ ]:
# Create directory for storing the base model, videos, and metrics
models_dir = "models/ppo/base"
videos_dir = "videos/ppo/base"
metrics_dir = "metrics"

os.makedirs(models_dir, exist_ok=True)
os.makedirs(videos_dir, exist_ok=True)
os.makedirs(metrics_dir, exist_ok=True)

In [ ]:
# Make and define training and evaluation environments (render mode is none because vizualizing significantly slows down training and evaluation)
def make_env(render_mode=None):
    env = gym.make(ENV_ID, render_mode=render_mode)
    env = Monitor(env)
    return env

train_env = make_env()
eval_env = make_env()

In [ ]:
# Improve efficiency with parallel environments
from stable_baselines3.common.env_util import make_vec_env
train_env = make_vec_env(lambda: make_env(), n_envs=4)

In [ ]:
# Train PPO for 1_000_000 timesteps and save the training time in seconds
model = PPO(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=64,
    gamma=0.99,
    verbose=1,
)

start = time.time()
model.learn(
    total_timesteps=1_000_000,
    progress_bar=True,
)

training_time_seconds = time.time() - start

Using cuda device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Streaming output truncated to the last 5000 lines.
|    loss                 | 3.81        |
|    n_updates            | 170         |
|    policy_gradient_loss | -0.000745   |
|    std                  | 1.01        |
|    value_loss           | 15.1        |
-----------------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 728          |
|    ep_rew_mean          | -236         |
| time/                   |              |
|    fps                  | 853          |
|    iterations           | 19           |
|    time_elapsed         | 91           |
|    total_timesteps      | 77824        |
| train/                  |              |
|    approx_kl            | 0.0018041099 |
|    clip_fraction        | 0.00632      |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.42        |
|    explained_variance   | 0.916        |
|    learning_rate        | 0.0003       |
|    loss

In [ ]:
# Evaluate and capture average episode length and rewards over 20 episodes and save the results in a metrics directory as a csv file with the training time in seconds, mean reward, std reward, mean episode timesteps, and std episode timesteps.
# Also print these metrics to the console.
episode_rewards, episode_lengths = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=20,
    deterministic=True,
    return_episode_rewards=True,
)

mean_reward = sum(episode_rewards) / len(episode_rewards)
std_reward = pd.Series(episode_rewards).std()

mean_episode_timesteps = sum(episode_lengths) / len(episode_lengths)
std_episode_timesteps = pd.Series(episode_lengths).std()

print("Mean reward:", mean_reward)
print("Std reward:", std_reward)
print("Mean episode timesteps:", mean_episode_timesteps)
print("Std episode timesteps:", std_episode_timesteps)
print("Training time seconds:", training_time_seconds)

Mean reward: 369.24337660000003
Std reward: 3.8898725163329377
Mean episode timesteps: 1123.55
Std episode timesteps: 40.190958658568185
Training time seconds: 1220.4390835762024


In [ ]:
# Save the base model
model_path = f"{models_dir}/ppo_base_1M"
model.save(model_path)

print(f"Saved model to: {model_path}.zip")

Saved model to: models/ppo/base/ppo_base_1M.zip


In [ ]:
# Save metrics
metrics = pd.DataFrame([{
    "env_id": ENV_ID,
    "model": "PPO",
    "training_timesteps": 1_000_000,
    "n_eval_episodes": 20,
    "mean_reward": mean_reward,
    "std_reward": std_reward,
    "mean_episode_timesteps": mean_episode_timesteps,
    "std_episode_timesteps": std_episode_timesteps,
    "training_time_seconds": training_time_seconds,
}])

metrics_path = f"{metrics_dir}/ppo_base_metrics.csv"

if os.path.exists(metrics_path):
    old = pd.read_csv(metrics_path)
    metrics = pd.concat([old, metrics], ignore_index=True)

metrics.to_csv(metrics_path, index=False)
metrics

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,env_id,model,training_timesteps,n_eval_episodes,mean_reward,std_reward,mean_episode_timesteps,std_episode_timesteps,training_time_seconds
0,SailboatDiscrete-v0,PPO,100000,20,-199.105619,58.076113,452.15,452.323973,247.131226
1,Sailboat-v0,PPO,100000,20,369.243377,3.889873,1123.55,40.190959,1220.439084
2,Sailboat-v0,PPO,1000000,20,369.243377,3.889873,1123.55,40.190959,1220.439084


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Record video an evaluation episode of the trained model and save it in the videos directory with a name prefix of ppo_base_trained. The video should be saved as a mp4 file and should be named ppo_base_trained_episode_0.mp4.
video_env = gym.make(ENV_ID, render_mode="rgb_array")
video_env = RecordVideo(
    video_env,
    video_folder=videos_dir,
    name_prefix="ppo_base_trained",
    episode_trigger=lambda episode_id: episode_id == 0,
)

obs, info = video_env.reset()
done = False
truncated = False

while not (done or truncated):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = video_env.step(action)

import builtins # imported to fix bug with recording video
builtins.quit = lambda *args, **kwargs: None # imported to fix bug with recording video
video_env.close()


/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content/drive/MyDrive/tiny_tackers/videos/ppo/base folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezon

- - - - - - - - - - - - - - - - - - - -
# Transfer Learning Step 1
## Train the base model to round a windward buoy and move to the next target
- Windward Buoy
- Reach Buoy
- - - - - - - - - - - - - - - - - - - -


In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Confirm my repo location
%cd "/content/drive/My Drive/tiny_tackers"
!pwd

/content/drive/My Drive/tiny_tackers
/content/drive/My Drive/tiny_tackers


In [3]:
# Install Stable Baselines3 library if not already installed
!pip install stable-baselines3 gymnasium pygame

In [4]:
# Confirm necessary libraries and imports are installed if they weren't already
import os
import sys
import time
import pandas as pd
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env
from gymnasium.wrappers import RecordVideo

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
# Set Path to the Race Environment
RACE_ENV_PATH = "/content/drive/My Drive/tiny_tackers/gym_sailing_environments/gym_sailing_race"
sys.path.append(RACE_ENV_PATH)

In [6]:
# Import race environment and ensure it's coming from the correct path. Ending in tiny_tackers/gym_sailing_environments/gym_sailing_race/gym_sail_race/__init__.py
import gym_sail_race
print(gym_sail_race.__file__)

/content/drive/My Drive/tiny_tackers/gym_sailing_environments/gym_sailing_race/gym_sail_race/__init__.py


In [7]:
# Specify that the environment is going to be one where continuous actions are possible and it's the 2 mark race objective
ENV_ID = "SailboatRace2Mark-v0"

In [8]:
# Make folders for outcomes if folders do not already exist. Define new paths so we don't overwrite base model results or zip file
models_dir = "models/ppo/base_to_2mark_race"
videos_dir = "videos/ppo/base_to_2mark_race"
metrics_dir = "metrics"

os.makedirs(models_dir, exist_ok=True)
os.makedirs(videos_dir, exist_ok=True)
os.makedirs(metrics_dir, exist_ok=True)

In [9]:
# Make and define training and evaluation environments (render mode is none because vizualizing significantly slows down training and evaluation)
def make_env(render_mode=None):
    env = gym.make(
        ENV_ID,
        render_mode=render_mode,
    )
    env = Monitor(env)
    return env


In [10]:
# Improve efficiency with parallel environments in training and make train env
from stable_baselines3.common.env_util import make_vec_env
train_env = make_vec_env(lambda: make_env(), n_envs=4)

/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google.cloud')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-pa

In [11]:
# Make eval environment
eval_env = make_env()

In [12]:
# Load the base model if you completed base model training from the steps above
model = PPO.load("models/ppo/base/ppo_base_1M.zip", env=train_env)

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [13]:
# Train the base model for an additional 1M steps in the race environment, where it needs to hit 2 marks
start = time.time()

model.learn(
    total_timesteps=1_000_000,
    progress_bar=True,
)

training_time_seconds = time.time() - start

Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

Streaming output truncated to the last 5000 lines.
|    loss                 | 5.4         |
|    n_updates            | 2620        |
|    policy_gradient_loss | -0.00174    |
|    std                  | 0.392       |
|    value_loss           | 18.1        |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1.43e+03    |
|    ep_rew_mean          | -93.6       |
| time/                   |             |
|    fps                  | 795         |
|    iterations           | 19          |
|    time_elapsed         | 97          |
|    total_timesteps      | 77824       |
| train/                  |             |
|    approx_kl            | 0.011700388 |
|    clip_fraction        | 0.0988      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.469      |
|    explained_variance   | 0.964       |
|    learning_rate        | 0.0003      |
|    loss                

In [14]:
# Evaluate and capture average episode length and rewards over 20 episodes and save the results in a metrics directory as a csv file with the training time in seconds, mean reward, std reward, mean episode timesteps, and std episode timesteps.
# Also print these metrics to the console.
episode_rewards, episode_lengths = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=20,
    deterministic=True,
    return_episode_rewards=True,
)

mean_reward = sum(episode_rewards) / len(episode_rewards)
std_reward = pd.Series(episode_rewards).std()

mean_episode_timesteps = sum(episode_lengths) / len(episode_lengths)
std_episode_timesteps = pd.Series(episode_lengths).std()

print("Mean reward:", mean_reward)
print("Std reward:", std_reward)
print("Mean episode timesteps:", mean_episode_timesteps)
print("Std episode timesteps:", std_episode_timesteps)
print("Training time seconds:", training_time_seconds)

Mean reward: 556.1588681000001
Std reward: 87.17929409245636
Mean episode timesteps: 1287.1
Std episode timesteps: 45.16857314549574
Training time seconds: 1303.0308735370636


In [15]:
# Save the base to 2mark race model
model_path = f"{models_dir}/ppo_base_to_2mark_race_2M"
model.save(model_path)
print(f"Saved model to: {model_path}.zip")

Saved model to: models/ppo/base_to_2mark_race/ppo_base_to_2mark_race_2M.zip


In [16]:
# Save metrics
metrics = pd.DataFrame([{
    "env_id": ENV_ID,
    "model": "PPO",
    "source_model": "models/ppo/base/ppo_base_1M.zip",
    "training_stage": "base_to_race",
    "additional_training_timesteps": 500_000,
    "n_eval_episodes": 20,
    "mean_reward": mean_reward,
    "std_reward": std_reward,
    "mean_episode_timesteps": mean_episode_timesteps,
    "std_episode_timesteps": std_episode_timesteps,
    "training_time_seconds": training_time_seconds,
}])

metrics_path = f"{metrics_dir}/ppo_base_to_2mark_race_metrics.csv"

if os.path.exists(metrics_path):
    old = pd.read_csv(metrics_path)
    metrics = pd.concat([old, metrics], ignore_index=True)

metrics.to_csv(metrics_path, index=False)
metrics

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,env_id,model,source_model,training_stage,additional_training_timesteps,n_eval_episodes,mean_reward,std_reward,mean_episode_timesteps,std_episode_timesteps,training_time_seconds
0,SailboatRace-v0,PPO,models/ppo/base/ppo_base_1M.zip,base_to_race,500000,20,-4725.640139,NaN,20000.0,NaN,644.248418
1,SailboatRace2Mark-v0,PPO,models/ppo/base/ppo_base_1M.zip,base_to_race,500000,20,-105.396178,NaN,2607.0,NaN,662.536380
2,SailboatRace2Mark-v0,PPO,models/ppo/base/ppo_base_1M.zip,base_to_race,500000,20,556.158868,87.179294,1287.1,45.168573,1303.030874


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [17]:
# Record video an evaluation episode of the trained model and save it in the videos directory with a name prefix of ppo_base_trained. The video should be saved as a mp4 file and should be named ppo_base_trained_episode_0.mp4.

# Ensure that we are recording a video with the environment that only has two marks to go to
video_env = gym.make(
    ENV_ID,
    render_mode="rgb_array",
)

video_env = RecordVideo(
    video_env,
    video_folder=videos_dir,
    name_prefix="ppo_base_to_2mark_race_trained",
    episode_trigger=lambda episode_id: episode_id == 0,
)

obs, info = video_env.reset()
done = False
truncated = False

while not (done or truncated):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = video_env.step(action)

import builtins # imported to fix bug with recording video
builtins.quit = lambda *args, **kwargs: None # imported to fix bug with recording video
video_env.close()


/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content/drive/MyDrive/tiny_tackers/videos/ppo/base_to_2mark_race folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


- - - - - - - - - - - - - - - - - - - - - - - -
# Transfer Learning Step 2
## Training the base to race model on the full Race Environment

- Windward Buoy
- Reach Buoy
- Leeward Buoy
- Windward Buoy
- Leeward Buoy
- Windward Buoy
- - - - - - - - - - - - - - - - - - - - - - - -

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Confirm my repo location
%cd "/content/drive/My Drive/tiny_tackers"
!pwd

/content/drive/My Drive/tiny_tackers
/content/drive/My Drive/tiny_tackers


In [ ]:
# Install Stable Baselines3 library if not already installed
!pip install stable-baselines3 gymnasium pygame

In [ ]:
# Confirm necessary libraries and imports are installed if they weren't already
import os
import sys
import time
import pandas as pd
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env
from gymnasium.wrappers import RecordVideo

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Set Path to the Race Environment
RACE_ENV_PATH = "/content/drive/My Drive/tiny_tackers/gym_sailing_environments/gym_sailing_race"
sys.path.append(RACE_ENV_PATH)

In [ ]:
# Import race environment and ensure it's coming from the correct path. Ending in tiny_tackers/gym_sailing_environments/gym_sailing_race/gym_sail_race/__init__.py
import gym_sail_race
print(gym_sail_race.__file__)

/content/drive/My Drive/tiny_tackers/gym_sailing_environments/gym_sailing_race/gym_sail_race/__init__.py


In [ ]:
# Specifify that the environment is going to be one where continuous actions are possible and it's the race objective
ENV_ID = "SailboatRace-v0"

In [ ]:
# Test that we can run the environment
test_env = gym.make(ENV_ID)
obs, info = test_env.reset()
print("Observation:", obs)
print("Observation space:", test_env.observation_space)
print("Action space:", test_env.action_space)
action = test_env.action_space.sample()
obs, reward, terminated, truncated, info = test_env.step(action)
print("Step worked")
print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)
test_env.close()

/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google.cloud')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-pa

Observation: [ 0.         -0.19522105  0.          0.10705367  1.27899765]
Observation space: Box([-10.          -3.14159265  -1.          -3.14159265   0.        ], [ 10.           3.14159265   1.           3.14159265 100.        ], (5,), float64)
Action space: Box(-1.0, 1.0, (1,), float64)
Step worked
Reward: -0.11051600424876398
Terminated: False
Truncated: False


In [ ]:
# Make folders for outcomes if folders do not already exist. Define new paths so we don't overwrite base model results or zip file
models_dir = "models/ppo/race"
videos_dir = "videos/ppo/race"
metrics_dir = "metrics"

os.makedirs(models_dir, exist_ok=True)
os.makedirs(videos_dir, exist_ok=True)
os.makedirs(metrics_dir, exist_ok=True)

In [ ]:
# Make and define training and evaluation environments (render mode is none because vizualizing significantly slows down training and evaluation)
def make_env(render_mode=None):
    env = gym.make(ENV_ID, render_mode=render_mode)
    env = Monitor(env)
    return env

train_env = make_env()
eval_env = make_env()

In [ ]:
# Improve efficiency with parallel environments
from stable_baselines3.common.env_util import make_vec_env
train_env = make_vec_env(lambda: make_env(), n_envs=4)

In [ ]:
# Train PPO for 1_000_000 timesteps and save the training time in seconds
model = PPO(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=64,
    gamma=0.99,
    verbose=1,
)

start = time.time()
model.learn(
    total_timesteps=1_000_000,
    progress_bar=True,
)

training_time_seconds = time.time() - start

Using cuda device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: 
datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects 
to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Streaming output truncated to the last 5000 lines.
|    loss                 | 2.77        |
|    n_updates            | 170         |
|    policy_gradient_loss | -0.00122    |
|    std                  | 0.945       |
|    value_loss           | 12.9        |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1.32e+03    |
|    ep_rew_mean          | -381        |
| time/                   |             |
|    fps                  | 774         |
|    iterations           | 19          |
|    time_elapsed         | 100         |
|    total_timesteps      | 77824       |
| train/                  |             |
|    approx_kl            | 0.002879901 |
|    clip_fraction        | 0.0074      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.36       |
|    explained_variance   | 0.871       |
|    learning_rate        | 0.0003      |
|    loss                

In [ ]:
# Evaluate and capture average episode length and rewards over 20 episodes and save the results in a metrics directory as a csv file with the training time in seconds, mean reward, std reward, mean episode timesteps, and std episode timesteps.
# Also print these metrics to the console.
episode_rewards, episode_lengths = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=20,
    deterministic=True,
    return_episode_rewards=True,
)

mean_reward = sum(episode_rewards) / len(episode_rewards)
std_reward = pd.Series(episode_rewards).std()

mean_episode_timesteps = sum(episode_lengths) / len(episode_lengths)
std_episode_timesteps = pd.Series(episode_lengths).std()

print("Mean reward:", mean_reward)
print("Std reward:", std_reward)
print("Mean episode timesteps:", mean_episode_timesteps)
print("Std episode timesteps:", std_episode_timesteps)
print("Training time seconds:", training_time_seconds)

Mean reward: -309.696127
Std reward: 5.832011720969797e-14
Mean episode timesteps: 1130.0
Std episode timesteps: 0.0
Training time seconds: 1270.6828384399414


In [ ]:
# Save the RACE model
model_path = f"{models_dir}/ppo_race_1M"
model.save(model_path)

print(f"Saved model to: {model_path}.zip")

Saved model to: models/ppo/race/ppo_race_1M.zip


In [ ]:
# Save metrics
metrics = pd.DataFrame([{
    "env_id": ENV_ID,
    "model": "PPO",
    "training_timesteps": 1_000_000,
    "n_eval_episodes": 20,
    "mean_reward": mean_reward,
    "std_reward": std_reward,
    "mean_episode_timesteps": mean_episode_timesteps,
    "std_episode_timesteps": std_episode_timesteps,
    "training_time_seconds": training_time_seconds,
}])

metrics_path = f"{metrics_dir}/ppo_race_metrics.csv"

if os.path.exists(metrics_path):
    old = pd.read_csv(metrics_path)
    metrics = pd.concat([old, metrics], ignore_index=True)

metrics.to_csv(metrics_path, index=False)
metrics

NameError: name 'mean_reward' is not defined

In [ ]:
# Record video an evaluation episode of the trained model and save it in the videos directory with a name prefix of ppo_base_trained. The video should be saved as a mp4 file and should be named ppo_base_trained_episode_0.mp4.
video_env = gym.make(ENV_ID, render_mode="rgb_array")
video_env = RecordVideo(
    video_env,
    video_folder=videos_dir,
    name_prefix="ppo_race_trained",
    episode_trigger=lambda episode_id: episode_id == 0,
)

obs, info = video_env.reset()
done = False
truncated = False

while not (done or truncated):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = video_env.step(action)

import builtins # imported to fix bug with recording video
builtins.quit = lambda *args, **kwargs: None # imported to fix bug with recording video
video_env.close()


/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content/drive/My Drive/tiny_tackers/videos/ppo/race folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


NameError: name 'model' is not defined

In [ ]:
#Debugging

In [ ]:
video_env = gym.make(ENV_ID, render_mode="rgb_array")
obs, info = video_env.reset()

frame = video_env.render()

print(type(frame))
print(frame.shape)
print(frame.dtype)

video_env.close()

<class 'numpy.ndarray'>
(680, 680, 3)
uint8


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
env = gym.make(ENV_ID)

for i in range(5):
    obs, info = env.reset()
    print(f"Reset {i}")
    print("obs:", obs)
    print("info:", info)
    print()

env.close()

Reset 0
obs: [0.         0.19269012 0.         0.16656179 1.25410869]
info: {}

Reset 1
obs: [0.         0.18983249 0.         0.13700912 1.15626516]
info: {}

Reset 2
obs: [ 0.         -0.14745744  0.          0.12127385  1.19066536]
info: {}

Reset 3
obs: [0.         0.19071979 0.         0.14866379 1.21417877]
info: {}

Reset 4
obs: [ 0.         -0.17819893  0.          0.140121    1.24451635]
info: {}

